# Cleaning and fixing the missing data

#### Cleaning the missing data and calculate the data that need for further processing.
- Located at: Bushehr, Iran
- spreading factor 7
- bandwidth 125 kHz
- coding rate 4/5
- 868 MHz channel

<table>
  <tr>
    <th colspan="3"> Label and Terrain Penalty (Adjust terrain penalty with your own) </th>
  </tr>
  <tr>
    <th>Label</th>
    <th>Code</th>
    <th>Terrain Penalty</th>
  </tr>
  <tr>
  <!-- first row -->
  <tr>
    <td>Trees</td>
    <td>10</td>
    <td>0.6</td>
  </tr>
  <!-- second row -->
  <tr>
    <td>Shrubland</td>
    <td>20</td>
    <td>0.4</td>
  </tr>
  <!-- third row -->
  <tr>
    <td>Grassland</td>
    <td>30</td>
    <td>0.15</td>
  </tr>
  <!-- fourth row -->
  <tr>
    <td>Cropland</td>
    <td>40</td>
    <td>0.25</td>
  </tr>
  <!-- fifth row -->
  <tr>
    <td>Built-up</td>
    <td>50</td>
    <td>0.8</td>
  </tr>
  <!-- sixth row -->
  <tr>
    <td>Bare/ sparse vegetation</td>
    <td>60</td>
    <td>0.2</td>
  </tr>
  <!-- seventh row -->
  <tr>
    <td>Snow and ice</td>
    <td>70</td>
    <td>0.55</td>
  </tr>
  <!-- eighth row -->
  <tr>
    <td>Permanent water bodies</td>
    <td>80</td>
    <td>0.0</td>
  </tr>
  <!-- ninth row -->
  <tr>
    <td>Herbaceous wetlands</td>
    <td>90</td>
    <td>0.35</td>
  </tr>
  <!-- tenth row -->
  <tr>
    <td>Manggroves</td>
    <td>95</td>
    <td>0.5</td>
  </tr>
  <!-- eleventh row -->
  <tr>
    <td>Moss and lichen</td>
    <td>100</td>
    <td>0.2</td>
  </tr>
</table>


### Preprocessing data

This notebook contains the code to preprocess the raw data for the ABC2026 project.
Code to calculate PDR with SNR threshold and SF with more theoretical sound and accurate way. Also calculate the path loss with observed path loss and theoretical path loss.
Calculate distance between two points using geodesic distance.

In [1]:
import pandas as pd
import numpy as np
import ee
import os
from dotenv import load_dotenv
from datetime import datetime
from geopy.distance import geodesic  # For accurate distance calculation
from geographiclib.geodesic import Geodesic
import warnings
warnings.filterwarnings('ignore')

# Load environment variables from .env file
load_dotenv()

True

#### 1. Initialize Earth Engine

In [2]:
# Ensure you have authenticated using 'earthengine authenticate'
ee.Authenticate()  # Run once if not already authenticated

# Initialize Earth Engine
try:
    ee.Initialize(project=os.getenv('GEE_PROJECT_ID'))
    print("Google Earth Engine initialized successfully")
except Exception as e:
    print(f"Google Earth Engine initialization failed: {str(e)}")
    exit()

Google Earth Engine initialized successfully


#### 2. Define Constants (Adjust as Needed)

In [3]:
# Gateway coordinates
GATEWAYS = {
    'Gateway_24e124fffef07103': {'lat': 28.992878, 'lon': 50.841090},
    'Gateway_24e124fffef06fd1': {'lat': 28.98864,  'lon': 50.83648},
    'Gateway_e45f01fffe376ce6': {'lat': 28.9958761, 'lon': 50.8319432}
}

# Destination point (harbor area)
DESTINATION_COORDS = {'lat': 28.986, 'lon': 50.840}

# Land cover labels
LAND_COVER_LABELS = {
    10: 'Tree',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / sparse vegetation',
    70: 'Snow and ice',
    80: 'Permanent water bodies',
    90: 'Herbaceous wetland',
    95: 'Mangroves',
    100: 'Moss and lichen'
}

# Terrain penalty factors (adjust as needed)
PENALTY_MAP = {
    10: 0.6,   # Tree cover - HIGH penalty (dense foliage blocks RF)
    20: 0.4,   # Shrubland - MODERATE penalty
    30: 0.15,  # Grassland - LOW penalty (minimal obstruction)
    40: 0.25,  # Cropland - LOW-MODERATE penalty
    50: 0.8,  # Built-up - VERY HIGH penalty (buildings = major obstruction)
    60: 0.2,   # Bare/sparse vegetation - LOW penalty
    70: 0.55,   # Snow and ice - MODERATE-HIGH penalty (reflection/absorption)
    80: 0.0,   # Permanent water bodies - NO penalty (best for propagation)
    90: 0.35,   # Herbaceous wetland - MODERATE-HIGH penalty
    95: 0.5,   # Mangroves - HIGH penalty (dense vegetation)
    100: 0.2   # Moss and lichen - VERY LOW penalty
}

# SNR threshold for LoRa (adjust as needed)
SNR_THRESHOLD = {
    7: -7.5,  # SF7
    8: -10,   # SF8
    9: -12.5, # SF9
    10: -15,  # SF10
    11: -17.5, # SF11
    12: -20   # SF12
}

# affect the PDR, adjust as needed and based on enivronmental conditions
# Map land cover to k value (empirically tuned)
LAND_COVER_TO_K = {
    10: 0.25,  # Tree cover → dense foliage
    20: 0.28,  # Shrubland
    30: 0.32,  # Grassland
    40: 0.33,  # Cropland
    50: 0.20,  # Built-up → severe multipath
    60: 0.40,  # Bare/sparse → near-LOS
    70: 0.35,  # Snow/ice → reflective but open
    80: 0.45,  # Water → excellent for LoRa (over ocean)
    90: 0.22,  # Wetland
    95: 0.20,  # Mangroves → worst case
    100: 0.30  # Moss/lichen (tundra)
}

# LoRa parameters
FREQUENCY_MHZ = 868
TX_POWER = 14
SPREADING_FACTOR = 7

#### 3. Load and Clean the Raw Data

In [4]:
print("Loading raw data...")
df_raw = pd.read_excel(r'../raw_data/data_1.xlsx', sheet_name='Sheet1')

# Basic inspection
print("\nInitial Data Shape:", df_raw.shape)
print("\nInitial Data Info:")
print(df_raw.info())
print("\nInitial Data Head:")
print(df_raw.head())

# Clean data: Remove rows where Lat/Lon are 0 or missing, or where all RSSI values are 0
def is_valid_location(row):
    # Check if Lat and Lon are not 0 and are not NaN
    return pd.notna(row['Lot']) and pd.notna(row['Lon']) and row['Lot'] != 0 and row['Lon'] != 0

# Apply the location filter
df_clean = df_raw[df_raw.apply(is_valid_location, axis=1)].copy()

print(f"\nData shape after location cleaning: {df_clean.shape}")

# Identify RSSI columns (those starting with 'RSSI(') and not the problematic one)
rssi_cols = [col for col in df_clean.columns if col.startswith('RSSI(') and '(b827eb5555e594df)' not in col]
print(f"\nDetected RSSI columns: {rssi_cols}")

# Focus on the primary gateway mentioned in the PDF and example script: 'RSSI (24e124fffef07103)'
primary_rssi_col = 'RSSI (24e124fffef07103)'
# Find the corresponding SNR column (it's the one immediately after the primary RSSI)
primary_snr_col = df_clean.columns[df_clean.columns.get_loc(primary_rssi_col) + 1]
print(f"Corresponding SNR column for {primary_rssi_col}: {primary_snr_col}")

# Filter rows where the primary RSSI is not 0, -999, or NaN
df_clean = df_clean[(df_clean[primary_rssi_col] != 0) & (df_clean[primary_rssi_col] != -999) & pd.notna(df_clean[primary_rssi_col])]

print(f"\nData shape after filtering for primary RSSI ({primary_rssi_col}): {df_clean.shape}")

# Select relevant columns including the corresponding SNR
df_proc = df_clean[['Time', 'Lot', 'Lon', 'DEVID', primary_rssi_col, primary_snr_col]].copy()
df_proc.columns = ['Time', 'latitude', 'longitude', 'DEVID', 'RSSI', 'SNR']  # Rename for clarity

# Convert Time to datetime if needed
df_proc['Time'] = pd.to_datetime(df_proc['Time'])

Loading raw data...

Initial Data Shape: (1488, 14)

Initial Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1488 entries, 0 to 1487
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Time                     1488 non-null   object 
 1   Lot                      1488 non-null   float64
 2   Lon                      1488 non-null   float64
 3   DEVID                    1488 non-null   object 
 4   RSSI(b827eb5555e594df)   1488 non-null   int64  
 5   SNR                      1488 non-null   float64
 6   RSSI (24e124fffef07103)  1488 non-null   int64  
 7   SNR                      1488 non-null   float64
 8   RSSI(24e124fffef06f9e)   1488 non-null   int64  
 9   SNR.1                    1488 non-null   float64
 10  RSSI(24e124fffef06fd1)   1488 non-null   int64  
 11  SNR.2                    1488 non-null   float64
 12  RSSI(e45f01fffe376ce6)   1488 non-null   int64  
 13  SNR.3 

#### 4. Define Functions for Feature Calculation

In [5]:
def calculate_observed_attenuation(rssi, tx_power_dbm=TX_POWER):
    """Calculate total observed signal attenuation (in dB)."""
    return tx_power_dbm - rssi

def calculate_theoretical_path_loss(distance_m, freq_mhz=FREQUENCY_MHZ, land_cover_code=None):
    """
    Calculate theoretical path loss using Free-Space Path Loss (FSPL) 
    plus environment-specific excess loss (in dB).
    """
    if distance_m <= 0:
        return np.inf
    
    # Clamp distance to avoid log(0)
    distance_m = max(distance_m, 1.0)

    # Free Space Path Loss (FSPL) calculation
    # PL = 20 * log10(d) + 20 * log10(f) + 20 * log10(4 * π / c)
    # Formula: FSPL = 20 * log10(distance_m) + 20 * log10(freq_mhz) - 27.55
    # - distance_m: distance between transmitter and receiver in meters
    # - freq_mhz: frequency of the signal in MHz
    # - The constant -27.55 is derived from:
    #   20 * log10(4 * π / c) + 20 * log10(1e6)
    #   where:
    #     - c = 3e8 m/s (speed of light)
    #     - 1e6 = 10^6 (conversion from Hz to MHz)
    #   Calculated as:
    #     20 * log10(4 * π / 3e8) = -147.54 dB
    #     20 * log10(1e6) = +120 dB   ->  from Hz to MHz
    #     Total = -147.54 + 120 = -27.54 dB (≈ -27.55 dB)
    # Result is in decibels (dB)
    fspl = 20 * np.log10(distance_m) + 20 * np.log10(freq_mhz) - 27.55

    # Excess loss (dB) based on land cover — tuned for harbor/industrial areas
    excess_loss_db = {
        10: 12,  # Tree
        20: 10,  # Shrubland
        30: 6,   # Grassland
        40: 5,   # Cropland
        50: 20,  # Built-up (harbor: metal containers, cranes)
        60: 4,   # Bare/sparse
        70: 8,   # Snow/ice
        80: 5,   # Water
        90: 11,  # Wetland
        95: 15,  # Mangroves
        100: 7   # Moss/lichen
    }.get(land_cover_code, 12)  # Default: urban-like

    return fspl + excess_loss_db

def calculate_elevation(lat, lon):
    """Fetch elevation using Earth Engine."""
    point = ee.Geometry.Point([lon, lat])
    elevation_dataset = ee.Image('USGS/SRTMGL1_003')
    try:
        elevation_value = elevation_dataset.sample(region=point, scale=30).first().get('elevation').getInfo()
        return elevation_value if elevation_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching elevation for {lat}, {lon}: {e}")
        return np.nan

def calculate_land_cover(lat, lon):
    """Fetch land cover code using Earth Engine."""
    point = ee.Geometry.Point([lon, lat])
    landcover_dataset = ee.ImageCollection('ESA/WorldCover/v200').first()
    try:
        landcover_value = landcover_dataset.sample(region=point, scale=10).first().get('Map').getInfo()
        return landcover_value if landcover_value is not None else np.nan
    except Exception as e:
        print(f"Error fetching land cover for {lat}, {lon}: {e}")
        return np.nan

def land_cover_code_to_label(code):
    """Convert land cover code to label."""
    return LAND_COVER_LABELS.get(code, f'Unknown ({code})')

def determine_terrain_penalty(land_cover_code):
    """Map land cover to penalty factor (for interpretability only)."""
    return PENALTY_MAP.get(land_cover_code, 0.8)

def snr_to_pdr(snr, spreading_factor=7, land_cover_code=None):
    """Convert SNR to PDR for LoRa."""  
    snr_threshold = SNR_THRESHOLD.get(spreading_factor, -7.5)
    margin = snr - snr_threshold
    
    if margin <= 0:
        return 0.0
    
    # Default k if land cover unknown
    k = LAND_COVER_TO_K.get(land_cover_code, 0.3)  # conservative default
    
    pdr = 1 - np.exp(-k * margin)
    return max(0.0, min(1.0, pdr))

def get_path_spatial_features(lat1, lon1, lat2, lon2, num_samples=15):
    if num_samples < 2:
        num_samples = 2
        
    geod = Geodesic.WGS84
    g = geod.Inverse(lat1, lon1, lat2, lon2)
    distance = g['s12']
    azi1 = g['azi1']
    
    points = []
    for i in range(num_samples):
        frac = i / (num_samples - 1)
        g_point = geod.Direct(lat1, lon1, azi1, frac * distance)
        points.append((g_point['lat2'], g_point['lon2']))
    
    # Rest of the function remains the same...
    lc_img = ee.ImageCollection("ESA/WorldCover/v200").first()
    srtm = ee.Image('USGS/SRTMGL1_003')
    
    lc_codes = []
    elevs = []
    
    for pt_lat, pt_lon in points:
        try:
            pt = ee.Geometry.Point([pt_lon, pt_lat])
            
            # Land cover
            lc_val = lc_img.sample(region=pt, scale=10).first()
            if lc_val:
                code = lc_val.get('Map').getInfo()
                if code is not None:
                    lc_codes.append(int(code))
            
            # Elevation
            elev_val = srtm.sample(region=pt, scale=30).first()
            if elev_val:
                elev = elev_val.get('elevation').getInfo()
                if elev is not None:
                    elevs.append(float(elev))
        except Exception:
            continue  # Skip failed points
    
    if not lc_codes or not elevs:
        return _nan_path_features()
    
    # Compute features
    total = len(lc_codes)
    built_up_frac = sum(1 for c in lc_codes if c == 50) / total
    veg_frac = sum(1 for c in lc_codes if c in {10, 20, 90, 95}) / total
    water_frac = sum(1 for c in lc_codes if c == 80) / total
    
    from collections import Counter
    dominant_lc = Counter(lc_codes).most_common(1)[0][0]
    avg_penalty = np.mean([PENALTY_MAP.get(c, 0.8) for c in lc_codes])
    elev_std = np.std(elevs) if len(elevs) > 1 else 0.0
    max_obstruction = float(np.max(elevs) - np.min(elevs))
    
    return {
        'dominant_lc': dominant_lc,
        'built_up_frac': built_up_frac,
        'veg_frac': veg_frac,
        'water_frac': water_frac,
        'avg_penalty': avg_penalty,
        'elev_std': elev_std,
        'max_obstruction': max_obstruction
    }

def _nan_path_features():
    return {k: np.nan for k in ['dominant_lc', 'built_up_frac', 'veg_frac', 'water_frac', 
                                'avg_penalty', 'elev_std', 'max_obstruction']}
    
def calculate_distance_to_gateways(lat, lon, gateways_dict):
    """Calculate distance to closest gateway."""
    distances = {}
    for gw_id, coords in gateways_dict.items():
        dist = geodesic((lat, lon), (coords['lat'], coords['lon'])).meters
        distances[gw_id] = dist
    if not distances:
        return np.nan, None
    min_dist = min(distances.values())
    closest_id = [k for k, v in distances.items() if v == min_dist][0]
    return min_dist, closest_id

#### 5. Data Preparation and Preprocessing

In [6]:
print(f"\nCalculating features for {len(df_proc)} data points...")

# Initialize lists
elevations = []
land_covers = []
observed_path_losses = []
theoretical_path_losses = []
excess_losses = []
pdrs = []
distances_to_start = []
distances_to_destination = []
closest_gw_ids = []
path_dominant_lcs = []
path_built_up_fracs = []
path_veg_fracs = []
path_water_fracs = []
path_avg_penalties = []
path_elev_stds = []
max_terrain_obstructions = []

for idx, row in df_proc.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    rssi = row['RSSI']
    snr = row['SNR']

    # Distances
    dist_to_start, closest_gw_id = calculate_distance_to_gateways(lat, lon, GATEWAYS)
    dist_to_dest = geodesic((lat, lon), (DESTINATION_COORDS['lat'], DESTINATION_COORDS['lon'])).meters

    closest_gw = GATEWAYS[closest_gw_id]
    path_feats = get_path_spatial_features(lat, lon, closest_gw['lat'], closest_gw['lon'])
    
    # Geospatial features
    elev = calculate_elevation(lat, lon)
    lc_code = calculate_land_cover(lat, lon)

    # Path loss calculations
    observed_pl = calculate_observed_attenuation(rssi)
    theoretical_pl = calculate_theoretical_path_loss(dist_to_start, FREQUENCY_MHZ, lc_code)
    excess_loss = observed_pl - theoretical_pl

    # PDR
    pdr = snr_to_pdr(snr, SPREADING_FACTOR, lc_code)

    # Append
    elevations.append(elev)
    land_covers.append(lc_code)
    observed_path_losses.append(observed_pl)
    theoretical_path_losses.append(theoretical_pl)
    excess_losses.append(excess_loss)
    pdrs.append(pdr)
    distances_to_start.append(dist_to_start)
    distances_to_destination.append(dist_to_dest)
    closest_gw_ids.append(closest_gw_id)
    path_dominant_lcs.append(path_feats['dominant_lc'])
    path_built_up_fracs.append(path_feats['built_up_frac'])
    path_veg_fracs.append(path_feats['veg_frac'])
    path_water_fracs.append(path_feats['water_frac'])
    path_avg_penalties.append(path_feats['avg_penalty'])
    path_elev_stds.append(path_feats['elev_std'])
    max_terrain_obstructions.append(path_feats['max_obstruction'])
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1} rows...")

# Add to DataFrame
df_proc['distance_to_start'] = distances_to_start
df_proc['distance_to_destination'] = distances_to_destination
df_proc['closest_gateway'] = closest_gw_ids
df_proc['elevation'] = elevations
df_proc['land_cover_code'] = land_covers
df_proc['land_cover_label'] = df_proc['land_cover_code'].apply(land_cover_code_to_label)
df_proc['observed_path_loss'] = observed_path_losses
df_proc['theoretical_path_loss'] = theoretical_path_losses
df_proc['excess_loss'] = excess_losses
df_proc['terrain_penalty'] = df_proc['land_cover_code'].apply(determine_terrain_penalty)
df_proc['PDR'] = pdrs
df_proc['latency_ms'] = 100.0  # Placeholder
df_proc['path_dominant_land_cover'] = path_dominant_lcs
df_proc['path_built_up_fraction'] = path_built_up_fracs
df_proc['path_vegetation_fraction'] = path_veg_fracs
df_proc['path_water_fraction'] = path_water_fracs
df_proc['path_avg_penalty'] = path_avg_penalties
df_proc['path_elevation_std'] = path_elev_stds
df_proc['max_terrain_obstruction_m'] =  max_terrain_obstructions


Calculating features for 1268 data points...
Processed 100 rows...
Processed 200 rows...
Processed 300 rows...
Processed 400 rows...
Processed 500 rows...
Processed 700 rows...
Processed 900 rows...
Processed 1000 rows...
Processed 1100 rows...
Processed 1200 rows...
Processed 1300 rows...
Processed 1400 rows...


#### 6. Final Data Preparation

In [7]:
print("\nProcessed Data Shape:", df_proc.shape)
print("\nProcessed Data Info:")
print(df_proc.info())
print("\nProcessed Data Head:")
print(df_proc.head())

# Check for any remaining NaN values in critical columns
critical_cols = [
    'latitude', 'longitude', 'RSSI', 'SNR', 'elevation', 'land_cover_code',
    'observed_path_loss', 'PDR', 'distance_to_start', 'distance_to_destination'
]
print(df_proc[critical_cols].isnull().sum())

# Drop rows where critical features couldn't be calculated (e.g., elevation/land_cover failed)
df_final = df_proc.dropna(subset=critical_cols)
print(f"\nFinal data shape after dropping NaNs: {df_final.shape}")


Processed Data Shape: (1268, 25)

Processed Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1268 entries, 7 to 1487
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Time                       1268 non-null   datetime64[ns]
 1   latitude                   1268 non-null   float64       
 2   longitude                  1268 non-null   float64       
 3   DEVID                      1268 non-null   object        
 4   RSSI                       1268 non-null   int64         
 5   SNR                        1268 non-null   float64       
 6   distance_to_start          1268 non-null   float64       
 7   distance_to_destination    1268 non-null   float64       
 8   closest_gateway            1268 non-null   object        
 9   elevation                  1268 non-null   int64         
 10  land_cover_code            1268 non-null   int64         
 11  land_cover_label  

#### 7. Save the Processed Dataset

In [9]:
# Reorder and rename
output_cols = [
    'latitude', 'longitude', 'elevation', 'land_cover_code', 'land_cover_label', 'terrain_penalty',
    'distance_to_start', 'distance_to_destination',
    'path_dominant_land_cover',
    'path_built_up_fraction',
    'path_vegetation_fraction',
    'path_water_fraction',
    'path_avg_penalty',
    'path_elevation_std',
    'max_terrain_obstruction_m',
    'RSSI', 'SNR', 'PDR', 'latency_ms',
    'observed_path_loss', 'theoretical_path_loss', 'excess_loss',
    'DEVID', 'Time', 'closest_gateway'
]

# Reorder and rename
df_output = df_final[output_cols].copy()
df_output.rename(columns={'land_cover_code': 'land_cover'}, inplace=True)

print("\nFinal Output Data Shape:", df_output.shape)
print("\nFinal Output Columns:", df_output.columns.tolist())
print("\nFinal Output Head:")
print(df_output.head())

# Save
output_filename = r'../data/processed_data_1.csv'
df_output.to_csv(output_filename, index=False)
print(f"\nProcessed dataset saved as '{output_filename}'")


Final Output Data Shape: (1268, 25)

Final Output Columns: ['latitude', 'longitude', 'elevation', 'land_cover', 'land_cover_label', 'terrain_penalty', 'distance_to_start', 'distance_to_destination', 'path_dominant_land_cover', 'path_built_up_fraction', 'path_vegetation_fraction', 'path_water_fraction', 'path_avg_penalty', 'path_elevation_std', 'max_terrain_obstruction_m', 'RSSI', 'SNR', 'PDR', 'latency_ms', 'observed_path_loss', 'theoretical_path_loss', 'excess_loss', 'DEVID', 'Time', 'closest_gateway']

Final Output Head:
     latitude  longitude  elevation  land_cover land_cover_label  \
7   28.989470  50.836233          3          50         Built-up   
8   28.989453  50.836219          3          50         Built-up   
9   28.989470  50.836188          3          50         Built-up   
10  28.989456  50.836189          3          50         Built-up   
11  28.989465  50.836227          3          50         Built-up   

    terrain_penalty  distance_to_start  distance_to_destinati